# TourGuide TTS/Translation Microservice

Ce notebook lance le microservice FastAPI sur Google Colab avec un GPU T4 gratuit.
Un tunnel ngrok expose le service pour que le Studio Web puisse l'appeler.

## Prérequis
1. **Runtime GPU** : Menu → Runtime → Change runtime type → **T4 GPU**
2. **Compte ngrok gratuit** : https://ngrok.com → récupère ton authtoken

## Étapes
1. Exécute chaque cellule dans l'ordre
2. Copie l'URL ngrok affichée
3. Colle-la dans ton `.env.local` : `NEXT_PUBLIC_MICROSERVICE_URL=https://xxxx.ngrok-free.app`
4. Le service tourne tant que le notebook est ouvert (~12h max)

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("[OK] GPU pret")
else:
    print("[FAIL] Pas de GPU ! Va dans Runtime > Change runtime type > T4 GPU")

## 2. Installer les dépendances

In [ ]:
# D'abord installer qwen-tts qui fixe sa propre version de transformers
!pip install -q qwen-tts
# Puis les autres deps (sans forcer transformers depuis git)
!pip install -q fastapi uvicorn soundfile pydub sentencepiece pyngrok requests nest_asyncio
!pip install -U flash-attn --no-build-isolation -q 2>/dev/null || echo "flash-attn optionnel"
!apt-get install -y -qq ffmpeg > /dev/null 2>&1

import transformers
import qwen_tts
print(f"transformers: {transformers.__version__}")
print("[OK] Dependances installees")

## 3. Configurer ngrok

Va sur https://dashboard.ngrok.com/get-started/your-authtoken et copie ton token.

In [ ]:
NGROK_AUTHTOKEN = ""  # @param {type:"string"}
MICROSERVICE_API_KEY = "tourguide-tts-2026"  # @param {type:"string"}

if not NGROK_AUTHTOKEN:
    print("\u274c Colle ton ngrok authtoken ci-dessus !")
    print("   https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    import os
    os.environ["MICROSERVICE_API_KEY"] = MICROSERVICE_API_KEY
    print(f"\u2705 API Key configur\u00e9e : {MICROSERVICE_API_KEY[:10]}...")
    print(f"\u2705 ngrok token configur\u00e9")

## 4. Charger les modèles TTS + Traduction

⏳ Première exécution : ~3-5 min (téléchargement modèles).
Les exécutions suivantes utilisent le cache.

In [ ]:
import torch
import logging
import time
import base64
import io
import re
import os
import tempfile
from urllib.parse import urlparse

import soundfile as sf
import requests as req_lib

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("tourguide")

# -- TTS : Qwen3-TTS 0.6B --
print("[...] Chargement de Qwen3-TTS 0.6B...")
start = time.time()

try:
    from qwen_tts import Qwen3TTSModel
    MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
    tts_model = Qwen3TTSModel.from_pretrained(
        MODEL_NAME,
        device_map="cuda:0",
        torch_dtype=torch.float16,
    )
    tts_mode = "qwen_tts"
    print(f"[OK] TTS charge via qwen_tts en {time.time()-start:.0f}s")
except Exception as e1:
    print(f"[FAIL] qwen_tts echoue : {e1}")
    tts_model = None
    tts_mode = None

tts_ready = tts_model is not None

# -- Discover available methods and speakers --
if tts_ready:
    try:
        speakers = tts_model.get_supported_speakers()
        print(f"   Speakers: {speakers}")
    except Exception as e:
        print(f"   get_supported_speakers: {e}")
        speakers = []

    try:
        langs = tts_model.get_supported_languages()
        print(f"   Languages: {langs}")
    except Exception as e:
        print(f"   get_supported_languages: {e}")
        langs = []

    methods = [m for m in dir(tts_model) if not m.startswith('_') and callable(getattr(tts_model, m))]
    print(f"   Methods: {methods}")

# -- Traduction : MarianMT (lazy) --
from transformers import MarianMTModel, MarianTokenizer

MARIAN_MODELS = {
    ("fr", "en"): "Helsinki-NLP/opus-mt-fr-en",
    ("fr", "it"): "Helsinki-NLP/opus-mt-fr-it",
    ("fr", "de"): "Helsinki-NLP/opus-mt-fr-de",
    ("fr", "es"): "Helsinki-NLP/opus-mt-fr-es",
}

translation_models = {}
translation_tokenizers = {}

def load_translation_pair(src, tgt):
    pair = (src, tgt)
    if pair in translation_models:
        return True
    model_name = MARIAN_MODELS.get(pair)
    if not model_name:
        return False
    print(f"[...] Chargement MarianMT {src} -> {tgt}...")
    translation_tokenizers[pair] = MarianTokenizer.from_pretrained(model_name)
    translation_models[pair] = MarianMTModel.from_pretrained(model_name)
    print(f"[OK] MarianMT {src} -> {tgt} charge")
    return True

print(f"\n[OK] Microservice pret (TTS: {tts_mode or 'OFF'} + traduction lazy)")
print(f"   VRAM utilisee : {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 5. Définir l'API FastAPI

In [ ]:
from google.colab import files

print(">> Upload ton sample vocal (.wav ou .mp3)")
print("   30 secondes minimum, 2-5 minutes ideal")
print("   Parle naturellement, comme si tu guidais quelqu'un")
print()

uploaded = files.upload()
REF_AUDIO_PATH = "/tmp/ref_voice_guide.wav"

# Convert to wav if needed
ref_filename = list(uploaded.keys())[0]
ref_size_kb = len(uploaded[ref_filename]) / 1024

if ref_filename.endswith(".wav"):
    with open(REF_AUDIO_PATH, "wb") as f:
        f.write(uploaded[ref_filename])
else:
    from pydub import AudioSegment as PydubSegment
    with open(f"/tmp/{ref_filename}", "wb") as f:
        f.write(uploaded[ref_filename])
    audio_seg = PydubSegment.from_file(f"/tmp/{ref_filename}")
    # Truncate to max 30s to avoid GPU OOM on T4
    if len(audio_seg) > 30000:
        print(f"   Audio trop long ({len(audio_seg)/1000:.0f}s), tronque a 30s")
        audio_seg = audio_seg[:30000]
    # Convert to mono 24kHz for Qwen3-TTS
    audio_seg = audio_seg.set_channels(1).set_frame_rate(24000)
    audio_seg.export(REF_AUDIO_PATH, format="wav")

info = sf.info(REF_AUDIO_PATH)
print(f"\n[OK] Sample vocal : {ref_filename} ({ref_size_kb:.0f} KB, {info.duration:.1f}s)")
print(f"   Format: {info.samplerate}Hz, {info.channels}ch")

# Transcription du sample vocal (le texte que tu as lu)
REF_TEXT = """Bienvenue a tous ! Je suis votre guide, et je vais vous accompagner a travers les ruelles de cette ville extraordinaire. Regardez autour de vous. Cette place, fondee au douzieme siecle, a traverse les epoques sans jamais perdre son charme. Les facades colorees que vous voyez datent du dix-huitieme siecle, ocre, jaune safran, rouge brique. Saviez-vous que ce batiment a accueilli trois rois de France ? Oui, trois ! Francois Premier en 1538, Henri Quatre en 1600, et Louis Quatorze en personne, en 1660. Ce quartier est celebre pour ses parfumeurs. Ici, on cultive le jasmin, la rose de mai, la tubereuse et la fleur d'oranger depuis plus de quatre cents ans. L'air que vous respirez en ce moment meme est charge de ces essences. Avez-vous des questions avant de continuer ? Non ? Alors allons-y, la suite du parcours nous reserve encore bien des surprises."""

print(f"   Ref text: {len(REF_TEXT)} chars")

# Create voice clone prompt for reuse
print("\n[...] Creation du voice prompt a partir de ta voix...")
try:
    ref_voice_prompt = tts_model.create_voice_clone_prompt(
        ref_audio=REF_AUDIO_PATH,
        ref_text=REF_TEXT,
    )
    print("[OK] Voice prompt cree ! Ta voix sera clonee sur toutes les generations.")
except Exception as e:
    ref_voice_prompt = None
    print(f"[WARN] Prompt echoue ({e}) -- on utilisera ref_audio + ref_text directement")

## 5a. Upload ton sample vocal

Upload un fichier `.wav` ou `.mp3` de ta voix (30s-5min).
Ce sample sera utilis\u00e9 comme r\u00e9f\u00e9rence pour le clonage vocal sur toutes les g\u00e9n\u00e9rations TTS.

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import traceback
import numpy as np

app = FastAPI(title="TourGuide TTS Microservice (Colab)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

API_KEY = os.environ.get("MICROSERVICE_API_KEY", "")

@app.middleware("http")
async def verify_api_key(request: Request, call_next):
    if request.url.path == "/health":
        return await call_next(request)
    if API_KEY and request.headers.get("X-API-Key") != API_KEY:
        return JSONResponse(status_code=401, content={"detail": "Invalid API key"})
    return await call_next(request)

class TTSRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=10000)
    language: str = Field(default="fr", pattern="^(fr|en|it|de|es|ja|ko|zh|ru)$")
    voice_id: str | None = None

class TranslateRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=50000)
    source_lang: str = Field(default="fr", pattern="^(fr|en|it|de|es)$")
    target_lang: str = Field(..., pattern="^(fr|en|it|de|es)$")

class SilenceRequest(BaseModel):
    audio_url: str = Field(..., min_length=1)

@app.get("/health")
async def health():
    has_voice = ref_voice_prompt is not None or os.path.exists(REF_AUDIO_PATH)
    return {
        "status": "ok",
        "tts": tts_ready,
        "tts_mode": tts_mode or "none",
        "voice_clone": has_voice,
        "translation": True,
        "silence_detection": True,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_used_gb": round(torch.cuda.memory_allocated()/1e9, 1),
    }

LANG_NAMES = {"fr": "french", "en": "english", "it": "italian", "de": "german", "es": "spanish",
              "ja": "japanese", "ko": "korean", "zh": "chinese", "ru": "russian"}

@app.post("/v1/tts/generate")
async def generate_tts(req: TTSRequest):
    if not tts_ready or not tts_model:
        return {"ok": False, "error": "TTS not ready"}
    if not os.path.exists(REF_AUDIO_PATH) and ref_voice_prompt is None:
        return {"ok": False, "error": "Pas de sample vocal. Relance la cellule 5a pour uploader ta voix."}
    try:
        sanitized = re.sub(r"<\|[^|]*\|>", "", req.text)
        lang = LANG_NAMES.get(req.language, "french")

        logger.info(f"TTS request: lang={lang}, text={sanitized[:80]}...")

        if ref_voice_prompt is not None:
            wavs, sr = tts_model.generate_voice_clone(
                text=sanitized,
                language=lang,
                voice_clone_prompt=ref_voice_prompt,
            )
        else:
            # Fallback: pass ref_audio + ref_text directly
            wavs, sr = tts_model.generate_voice_clone(
                text=sanitized,
                language=lang,
                ref_audio=REF_AUDIO_PATH,
                ref_text=REF_TEXT,
            )

        torch.cuda.empty_cache()

        audio_array = wavs[0] if isinstance(wavs, list) else wavs
        if hasattr(audio_array, 'cpu'):
            audio_array = audio_array.cpu().numpy()
        if isinstance(audio_array, np.ndarray) and audio_array.ndim > 1:
            audio_array = audio_array.squeeze()

        buf = io.BytesIO()
        sf.write(buf, audio_array, sr, format="WAV")
        buf.seek(0)
        audio_b64 = base64.b64encode(buf.read()).decode("ascii")
        duration_ms = int(len(audio_array) / sr * 1000)

        logger.info(f"TTS success: {duration_ms}ms audio, {len(audio_b64)//1024}KB base64")
        return {"ok": True, "audio_base64": audio_b64, "duration_ms": duration_ms}
    except Exception as e:
        torch.cuda.empty_cache()
        tb = traceback.format_exc()
        logger.error(f"TTS error: {e}\n{tb}")
        return {"ok": False, "error": f"{str(e)}"}

@app.post("/v1/translate/marianmt")
async def translate(req: TranslateRequest):
    if req.source_lang == req.target_lang:
        return {"ok": True, "translated_text": req.text}
    pair = (req.source_lang, req.target_lang)
    if not load_translation_pair(*pair):
        return {"ok": False, "error": f"Paire non supportee: {pair}"}
    try:
        tokenizer = translation_tokenizers[pair]
        model = translation_models[pair]
        inputs = tokenizer(req.text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            translated = model.generate(**inputs)
        result = tokenizer.decode(translated[0], skip_special_tokens=True)
        return {"ok": True, "translated_text": result}
    except Exception as e:
        logger.error(f"Translation error: {e}")
        return {"ok": False, "error": str(e)}

ALLOWED_HOSTS = {"s3.amazonaws.com", "s3.us-east-1.amazonaws.com"}
def is_allowed_url(url):
    try:
        parsed = urlparse(url)
        host = parsed.hostname or ""
        return host in ALLOWED_HOSTS or (host.endswith(".amazonaws.com") and ".s3." in host)
    except:
        return False

@app.post("/v1/silence-detect")
async def silence_detect(req: SilenceRequest):
    if not is_allowed_url(req.audio_url):
        return {"ok": False, "error": "URL non autorisee"}
    try:
        from pydub import AudioSegment
        from pydub.silence import detect_nonsilent
        resp = req_lib.get(req.audio_url, timeout=30, allow_redirects=False)
        resp.raise_for_status()
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp.write(resp.content)
            tmp_path = tmp.name
        try:
            audio = AudioSegment.from_file(tmp_path)
            segments = detect_nonsilent(audio, min_silence_len=800, silence_thresh=-40)
            if not segments:
                segments = [(0, len(audio))]
            return {"ok": True, "segments": [{"start_ms": s[0], "end_ms": s[1]} for s in segments]}
        finally:
            os.unlink(tmp_path)
    except Exception as e:
        logger.error(f"Silence detection error: {e}")
        return {"ok": False, "error": str(e)}

print("[OK] API FastAPI definie")
print("   TTS: generate_voice_clone (ta voix)")
print(f"   Langues: {list(LANG_NAMES.values())}")

## 6. Lancer le serveur + tunnel ngrok

Cette cellule tourne en continu. Le serveur reste actif tant que le notebook est ouvert.

**Copie l'URL ngrok** affichée et colle-la dans ton `.env.local` :
```
NEXT_PUBLIC_MICROSERVICE_URL=https://xxxx-xx-xx-xxx-xxx.ngrok-free.app
NEXT_PUBLIC_MICROSERVICE_API_KEY=tourguide-tts-2026
```

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok, conf

# Configurer ngrok
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Ouvrir le tunnel
public_url = ngrok.connect(8000, "http")

print("=" * 60)
print("[OK] MICROSERVICE EN LIGNE")
print("")
print(f"   URL publique : {public_url}")
print(f"   API Key      : {API_KEY}")
print("")
print(f"   Ajoute dans ton .env.local :")
print(f"   NEXT_PUBLIC_MICROSERVICE_URL={public_url}")
print(f"   NEXT_PUBLIC_MICROSERVICE_API_KEY={API_KEY}")
print("")
print(f"   Test : curl {public_url}/health")
print("=" * 60)

import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

## 7. Test rapide (optionnel)

Ouvre un autre onglet Colab ou utilise curl pour tester :

In [ ]:
# \u00c0 ex\u00e9cuter dans un AUTRE notebook ou terminal :
#
# curl -X POST https://VOTRE-URL.ngrok-free.app/v1/tts/generate \
#   -H "Content-Type: application/json" \
#   -H "X-API-Key: tourguide-tts-2026" \
#   -d '{"text": "Bienvenue sur la C\u00f4te d\u0027Azur.", "language": "fr"}'
#
# curl -X POST https://VOTRE-URL.ngrok-free.app/v1/translate/marianmt \
#   -H "Content-Type: application/json" \
#   -H "X-API-Key: tourguide-tts-2026" \
#   -d '{"text": "Bienvenue \u00e0 Nice.", "source_lang": "fr", "target_lang": "en"}'

print("Voir les commandes curl ci-dessus pour tester depuis un autre terminal.")